# The Anatomy of AI · Lecture 30a · Already Clean? I: Pointed or Round?
### The notebook

In 2025 Goodfire published an essay, *Neural Networks Are Already Clean*. Its claim: the neurons of a vision network
are, mostly, already clean detectors. A neuron that seems to fire on unrelated things (cats *and* foxes, cars *and*
dogs) is really detecting one thing, a shape or a pose, that those nouns share. In the lecture we tested that claim on
neurons this course has met. Here you rebuild the decisive tests yourself.

The notebook follows the lecture's plates in order, and each section names its plate. Wherever a result appears in the
lecture, a `check(...)` line compares yours with it and prints ✓. If a check fails, the notebook stops.

**Small versions.** The lecture's scans ran over all 1.28 million ImageNet training photos. Here we fetch only the photos
each test needs: a few hundred, by their number in ImageNet's validation set. Where a sample is smaller than the
lecture's, the check compares with the lecture's own scan of *the same photos*.

There are five **Your turn** exercises. Worked solutions are at the end.

It runs on free Colab in about 15 minutes on a **T4 GPU** (Runtime → Change runtime type → T4). No keys. It downloads
InceptionV1 (27 MB), about 3,500 small photos (about 60 MB) and, for the pose section, an orientation model (1.2 GB).

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, subprocess, sys
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "scipy==1.14.1", "pillow==11.1.0", "requests==2.32.3", "transformers==4.38.2")
pip("--no-deps", "torch-lucent==0.1.8")       # only its InceptionV1 port is used

In [ ]:
import io, json, math, os, time, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"       # the page shows results, not download bars
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=1e-9):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{'' if tol < 1e-6 or not isinstance(expected, (int, float)) else f' ± {tol:g}'})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

### Photos and the network

Two helpers. `fetch(ids)` downloads ImageNet-val photos by number, and `photo(i)` gives photo *i* as the network saw it:
256 × 256 pixels, centre-cropped to 224 × 224. Then InceptionV1, with one fix the whole course uses.

In [ ]:
# ImageNet's 50,000 validation photos, one at a time. The course used a public 256-pixel copy
# (benjamin-paine/imagenet-1k-256x256 on Hugging Face). Hugging Face's dataset viewer hands out any row by its number,
# so we fetch only the photos we need. Photo n here is photo n in the lecture.
import requests
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

ROWS = ("https://datasets-server.huggingface.co/rows?dataset=benjamin-paine/imagenet-1k-256x256"
        "&config=default&split=validation&offset={}&length={}")
os.makedirs("photos", exist_ok=True)


def _get(url, tries=10):
    """GET with patience: the viewer limits how fast one machine may ask, so back off when it says so (429)."""
    status = None
    for k in range(tries):
        try:
            r = requests.get(url, timeout=60)
            status = r.status_code
            if status == 200:
                return r
        except requests.RequestException as e:
            status = type(e).__name__
        time.sleep(min(60, 2 ** k) + np.random.rand())
    raise RuntimeError(f"could not fetch {url[:90]}... (last status {status})")


def _fetch_block(start, n):
    """Rows start..start+n-1 (at most 100) -> {row: (label, jpeg bytes)}, cached on disk."""
    need = [i for i in range(start, start + n) if not os.path.exists(f"photos/{i}.jpg")]
    if need:
        rows = _get(ROWS.format(start, n)).json()["rows"]
        def one(row):
            i = row["row_idx"]
            open(f"photos/{i}.jpg", "wb").write(_get(row["row"]["image"]["src"]).content)
            open(f"photos/{i}.label", "w").write(str(row["row"]["label"]))
        with ThreadPoolExecutor(8) as ex:
            list(ex.map(one, [r for r in rows if r["row_idx"] in need]))


def fetch(ids):
    """Download the photos with these numbers (and cache them)."""
    ids = sorted(set(int(i) for i in ids))
    todo = [i for i in ids if not os.path.exists(f"photos/{i}.jpg")]
    blocks = sorted({(i, 1) for i in todo})
    with ThreadPoolExecutor(3) as ex:              # a few row requests at a time
        list(ex.map(lambda b: _fetch_block(*b), blocks))


def fetch_range(start, n):
    """Download a contiguous run of photos, 100 rows per request."""
    for s in range(start, start + n, 100):          # one block of 100 rows at a time
        _fetch_block(s, min(100, start + n - s))


def photo(i, size=224):
    """Photo i as InceptionV1 saw it: 256 x 256, centre crop 224 x 224, RGB uint8."""
    im = Image.open(f"photos/{int(i)}.jpg").convert("RGB")
    if im.size != (256, 256):
        im = im.resize((256, 256), Image.BICUBIC)
    o = (256 - size) // 2
    return np.asarray(im.crop((o, o, o + size, o + size)), dtype=np.uint8)


def label(i):
    return int(open(f"photos/{int(i)}.label").read())

In [ ]:
# InceptionV1 (GoogLeNet, 2014), the network the essay and this lecture study, through lucent's PyTorch port.
# One fix first: the port's local response normalization used the wrong constants (lucent pull request #51, Gorton 2024).
# We correct the installed file before importing it, as every lecture in the course does.
import importlib, importlib.util, pathlib
_spec = importlib.util.find_spec("lucent")
_f = pathlib.Path(_spec.origin).parent / "modelzoo" / "inceptionv1" / "InceptionV1.py"
_s = _f.read_text()
_old = "size=9, alpha=9.99999974738e-05, beta=0.5, k=1"
if _old in _s:
    _f.write_text(_s.replace(_old, "size=10, alpha=10 * 0.00009999999747378752, beta=0.5, k=2"))
for _p in (_f.parent / "__pycache__").glob("InceptionV1*.pyc"):
    _p.unlink()                                   # a stale cached copy would keep the old constants
importlib.invalidate_caches()
assert "k=2" in _f.read_text(), "the normalization fix did not take"

import contextlib
from lucent.modelzoo import inceptionv1
with contextlib.redirect_stderr(io.StringIO()):          # torch.hub's download bar; one line below says it instead
    net = inceptionv1(pretrained=True).eval().to(DEV)
for p in net.parameters():
    p.requires_grad_(False)
print(f"InceptionV1 loaded ({sum(p.numel() for p in net.parameters()) / 1e6:.1f} million weights, a 27 MB download)")

BRANCHES = ["1x1", "3x3", "5x5", "pool_reduce"]


class Taps:
    """Record each layer's value *before* its ReLU, the four branches side by side in channel order,
    so 4d:295 is channel 295 of mixed4d. Before the ReLU both poles of a neuron are visible."""
    def __init__(self, layers):
        self.parts = {}
        for L in layers:
            for b in BRANCHES:
                getattr(net, f"{L}_{b}_pre_relu_conv").register_forward_hook(self._keep(L, b))

    def _keep(self, L, b):
        def f(_m, _i, out):
            self.parts[(L, b)] = out
        return f

    def __getitem__(self, L):
        return torch.cat([self.parts[(L, b)] for b in BRANCHES], 1)


def to_input(x01):
    """The port takes RGB in [-117, 138], as the original TensorFlow model did."""
    return x01 * 255.0 - 117.0


def batches(imgs, size=64):
    for s in range(0, len(imgs), size):
        x = torch.from_numpy(np.stack(imgs[s:s + size])).to(DEV).permute(0, 3, 1, 2).float() / 255
        yield to_input(x)

In [ ]:
# Which ImageNet classes are which. The 25 motor-vehicle classes of ImageNet-val, and the 118 dog breeds (151-268).
NAMES = requests.get("https://datasets-server.huggingface.co/info?dataset=benjamin-paine/imagenet-1k-256x256",
                     timeout=60).json()["dataset_info"]["default"]["features"]["label"]["names"]
NAMES = [n.split(",")[0].strip() for n in NAMES[:1000]]
VEHICLES = [407, 408, 436, 468, 511, 555, 569, 573, 575, 609, 627, 654, 656, 661, 665, 675, 717, 734, 751, 779, 803, 817, 864, 867, 874]
DOGS = set(range(151, 269))
print(len(VEHICLES), "vehicle classes, e.g.", [NAMES[c] for c in VEHICLES[:6]])

## Part A · 4e:55, pointed or round?

## Plate VI · Two poles

A **neuron** here is one channel of one layer: `mixed4e:55` is channel 55 of layer 4e. At every position of the photo
(a 14 × 14 grid) it has a value. We read it *before* the ReLU, so it can be negative as well as positive.

A neuron's **positive pole** is what drives its value highest; its **negative pole**, what drives it lowest. For one
photo we take the largest value over all positions (its *max*) and the smallest (its *min*).

The lecture found 4e:55's strongest photo in all of ImageNet-val to be photo 38200, a Pomeranian (+487.5), and its
weakest photo 13277, a pug (−245.0).

In [ ]:
taps = Taps(["mixed4c", "mixed4d", "mixed4e"])


def measure(ids, wanted):
    """Each photo's max and min, and its 14 x 14 (or 7 x 7) map, for each (layer, channel) in wanted."""
    fetch(ids)
    out = {w: {"max": [], "min": [], "map": []} for w in wanted}
    with torch.no_grad():
        for x in batches([photo(i) for i in ids]):
            net(x)
            for (L, c) in wanted:
                a = taps[L][:, c]
                out[(L, c)]["max"] += a.flatten(1).amax(1).tolist()
                out[(L, c)]["min"] += a.flatten(1).amin(1).tolist()
                out[(L, c)]["map"] += list(a.cpu().numpy())
    return {w: {k: (np.array(v) if k != "map" else v) for k, v in d.items()} for w, d in out.items()}


N55 = ("mixed4e", 55)
two = measure([38200, 13277], [N55])[N55]
check("4e:55's max on photo 38200, the Pomeranian", two["max"][0], 487.5, tol=10)
check("4e:55's min on photo 13277, the pug", two["min"][1], -245.0, tol=6)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
for a, i, m, t in zip(ax, [38200, 13277], two["map"], ["positive pole: its strongest photo", "negative pole: its weakest"]):
    a.imshow(photo(i))
    a.imshow(np.kron(m, np.ones((16, 16))), cmap="gray", alpha=0.45, extent=(0, 224, 224, 0))
    a.set_title(f"{t}\n{NAMES[label(i)]}, photo {i}", fontsize=10); a.axis("off")
plt.tight_layout(); plt.show()

**From scratch.** 4e:55 sits in layer 4e's 1 × 1 branch. So at each position its value is simply a weighted sum of the
528 numbers that layer 4d hands on (after 4d's ReLU), plus a bias: 528 weights and one bias in all. Here we do that sum
ourselves with NumPy and compare it with the network.

In [ ]:
conv = net.mixed4e_1x1_pre_relu_conv
print("channels in 4e's 1x1 branch:", conv.out_channels, "(so channel 55 is one of them)")
grab = {}
h = conv.register_forward_hook(lambda m, i, o: grab.update(x=i[0]))
with torch.no_grad():
    net(next(batches([photo(38200)])))
h.remove()
x = grab["x"][0].cpu().numpy()                       # 528 x 14 x 14: what 4d hands on
w = conv.weight[55, :, 0, 0].cpu().numpy()           # 528 weights
b = float(conv.bias[55])
by_hand = np.einsum("chw,c->hw", x, w) + b           # the weighted sum, at every position
network = taps["mixed4e"][0, 55].cpu().numpy()
check("largest difference, by hand vs the network", np.abs(by_hand - network).max(), 0, tol=0.05)
check("4e:55 on the Pomeranian, by hand", by_hand.max(), 487.5, tol=10)

## Plates VII–VIII · The test we wrote first

The essay describes 4e:55 as detecting **pointed** shapes, with **round** ones at its other pole. Before looking, the
lecture wrote down tests that description must pass. One was about cats: pointed-faced cats (Siamese and Egyptian cats,
100 photos) should beat round-faced cats (Persians, 50 photos).

The score is the **AUC**: the chance that a random pointed-faced cat scores higher than a random Persian. 0.5 is a coin
toss, 1 means every pointed face wins. The lecture measured **0.28**: the Persians win. The description failed its own
test. Below we fetch the 150 photos and count every pair ourselves.

In [ ]:
CATS = {"Persian": [129, 282, 732, 1039, 1100, 1131, 2035, 2966, 3381, 4410, 4893, 5626, 7807, 8490, 8985, 9593, 9898, 10217, 10994, 12797, 13784, 14215, 14812, 15371, 15897, 16193, 16353, 16597, 17813, 20764, 21702, 23826, 23897, 24426, 24499, 26829, 27905, 31179, 32608, 32909, 33194, 34243, 35171, 37732, 39389, 40779, 41346, 43897, 44096, 49995], "Siamese": [210, 2474, 2658, 3132, 4087, 4820, 5465, 10388, 10435, 11527, 11737, 12172, 15400, 17185, 17324, 17691, 20229, 20593, 20687, 22534, 24759, 25518, 25886, 26892, 28212, 29838, 30928, 32327, 34253, 34420, 35565, 36289, 36525, 36960, 38167, 39579, 39930, 39978, 40853, 43050, 43089, 43144, 43545, 43631, 43890, 45346, 45828, 46571, 47102, 48159], "Egyptian": [708, 1495, 2071, 4611, 4718, 5206, 5361, 7022, 8777, 9682, 10416, 10597, 11951, 13409, 13917, 14431, 14772, 16917, 19118, 21674, 22336, 23392, 23695, 23785, 23870, 23905, 23999, 25167, 26402, 26411, 27188, 27447, 28627, 28969, 29863, 31184, 32636, 33581, 37996, 38682, 38975, 40191, 40599, 41426, 41578, 42231, 44373, 46769, 46963, 49734]}
pointed = CATS["Siamese"] + CATS["Egyptian"]
persian = CATS["Persian"]
cats = measure(pointed + persian, [N55])[N55]["max"]
m_pointed, m_persian = cats[:len(pointed)], cats[len(pointed):]


def auc(a, b):
    """The chance that a random value from a beats a random value from b (a tie counts half)."""
    a, b = np.asarray(a)[:, None], np.asarray(b)[None, :]
    return float(((a > b) + 0.5 * (a == b)).mean())


print(f"{len(m_pointed) * len(m_persian)} pairs compared")
check("AUC, pointed-faced cats over Persians", auc(m_pointed, m_persian), 0.28, tol=0.02)
check("mean max, pointed-faced cats", m_pointed.mean(), 151.2, tol=4)
check("mean max, Persians", m_persian.mean(), 217.4, tol=5)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6))
for y, (v, name) in enumerate([(m_pointed, "Siamese + Egyptian (pointed)"), (m_persian, "Persian (round)")]):
    ax.scatter(v, np.full(len(v), y) + np.random.RandomState(y).uniform(-0.15, 0.15, len(v)), s=14,
               facecolor="none", edgecolor=INK, linewidth=0.8)
    ax.plot([v.mean()] * 2, [y - 0.3, y + 0.3], color=INK, lw=2)
ax.set_yticks([0, 1], ["pointed-faced", "Persian"]); ax.set_xlabel("4e:55, max over the photo")
ax.set_title("The Persians score higher: the pre-registered test fails", fontsize=10)
plt.tight_layout(); plt.show()

### Your turn 1 · The dog test

The lecture's second test: dogs with **pointed snouts** (collies, huskies, whippets …) should beat **flat-faced** dogs
(pugs, bulldogs, boxers …). With all 1,400 photos the lecture found AUC **0.691**: this one passes.

The photo numbers below are the first 10 validation photos of each breed. Measure 4e:55 on both groups and compute the
AUC.

*Hint: `measure(ids, [N55])[N55]["max"]`, then `auc(...)`.*

In [ ]:
POINTED_SNOUT_IDS = [1625, 3163, 3204, 4260, 4389, 5020, 5717, 6264, 6494, 9056, 3479, 3861, 4710, 6329, 7354, 7681, 9054, 11697, 12036, 14085, 671, 1899, 3302, 3733, 4765, 5394, 6337, 7631, 7836, 7896, 945, 1683, 2011, 4395, 4537, 4562, 5661, 7739, 7921, 9533, 133, 1857, 3682, 6585, 6674, 8176, 8342, 9003, 13032, 13296, 268, 992, 1313, 2282, 4191, 4966, 5063, 6538, 6583, 7217, 177, 2003, 2065, 5018, 5445, 5520, 6058, 7550, 8319, 9147, 728, 3744, 5574, 6051, 7215, 8089, 8269, 9013, 9415, 10499, 2, 179, 392, 1146, 1962, 1970, 3195, 3809, 3871, 4630, 262, 299, 1575, 3226, 3545, 5216, 5272, 6934, 7001, 8811, 21, 114, 227, 1136, 1249, 1547, 4433, 4775, 8005, 8378, 1343, 4803, 6534, 7694, 9092, 9223, 9467, 9809, 10685, 10969, 996, 1493, 1685, 1708, 3367, 3909, 3930, 7541, 8972, 9044, 1095, 1767, 2483, 3710, 4128, 5197, 5299, 5745, 5885, 6483, 1845, 2647, 2868, 3596, 4377, 4406, 4822, 4955, 7444, 7932, 1824, 2023, 2238, 2324, 4230, 5058, 5470, 6053, 6260, 7370]   # 10 photos of each of 16 pointed-snout breeds
FLAT_FACED_IDS = [219, 332, 1008, 1396, 1645, 5016, 7927, 8584, 10845, 11022, 349, 1296, 2405, 3353, 3773, 4673, 4769, 4940, 5456, 6313, 1312, 2779, 3854, 3937, 5125, 5326, 5676, 7317, 8193, 8626, 974, 2316, 2515, 2613, 3021, 3967, 4055, 4578, 4962, 7189, 1068, 2207, 2456, 3135, 4064, 7056, 7110, 7136, 7222, 9104, 906, 1243, 1392, 2700, 2768, 3840, 4048, 4080, 4358, 4556, 461, 1224, 1227, 1778, 3952, 4670, 6145, 7087, 9006, 9246, 634, 3521, 4448, 5262, 5494, 7000, 7294, 8336, 9188, 9552, 4505, 6241, 6485, 6956, 7233, 8195, 8213, 8966, 10728, 12741, 354, 535, 1151, 3695, 5909, 6415, 6536, 6663, 7653, 8956, 1480, 1763, 1954, 3376, 5238, 5254, 5684, 6584, 6608, 7496, 1567, 1924, 3326, 3690, 7546, 7725, 8334, 8432, 9434, 12445]      # 10 photos of each of 12 flat-faced breeds
# TODO: measure both groups and compute auc(pointed_snout, flat_faced)

## Plate IX · What a drawing shows

If 4e:55 detects pointed shapes, a drawing of a point should drive it. The lecture drew a **drop**: a disk with a tip
whose sharpness is set by *t*, from 0 (a circle) to 1 (a sharp teardrop), on a grey canvas. The height of the shape is
fixed, so size can't be the explanation. Here is the drawing code, unchanged from the lecture's.

In [ ]:
from PIL import ImageDraw


def drop(t, size=120, angle=0.0, polarity="dark", res=224, ss=4):
    """A drop on a mid-grey canvas. t = 0 a circle, t = 1 a sharp teardrop. angle 0 points the tip down."""
    R = res * ss
    bg, fg = (128, 128, 128), ((235, 235, 235) if polarity == "light" else (20, 20, 20))
    im = Image.new("RGB", (R, R), bg)
    d = ImageDraw.Draw(im)
    cx = cy = R / 2
    s = size * ss
    rot = math.radians(angle)
    P = lambda x, y: (cx + x * math.cos(rot) - y * math.sin(rot), cy + x * math.sin(rot) + y * math.cos(rot))
    r = s * 0.8 / (2.0 + 2.2 * t)            # the disk; the total height stays 0.8 x size
    D = r * (1.0 + 2.2 * t)                  # how far below the disk's centre the tip sits
    yc, yt = (r - D) / 2, (r + D) / 2
    phi = math.acos(min(1.0, r / D))
    pts = [P(0, yt)] + [P(r * math.cos(math.pi / 2 + phi + k / 240 * (2 * math.pi - 2 * phi)),
                          yc + r * math.sin(math.pi / 2 + phi + k / 240 * (2 * math.pi - 2 * phi))) for k in range(241)]
    d.polygon(pts, fill=fg)
    return np.asarray(im.resize((res, res), Image.LANCZOS))


def score(imgs):
    """4e:55's max on each drawing."""
    out = []
    with torch.no_grad():
        for x in batches(imgs):
            net(x)
            out += taps["mixed4e"][:, 55].flatten(1).amax(1).tolist()
    return np.array(out)


T = [round(v, 1) for v in np.linspace(0, 1, 11)]
sweep = score([drop(t, angle=270) for t in T])          # dark, tip pointing right
best = int(np.argmax(sweep))
check("sharpness of the best drawing", T[best], 0.8)
check("4e:55 on the best drawing", sweep[best], 114.05, tol=1.5)
check("best drawing / best photo", sweep[best] / 487.5, 0.234, tol=0.01)

In [ ]:
fig = plt.figure(figsize=(10, 3.4))
for k, t in enumerate(T):
    a = fig.add_subplot(2, 11, k + 1); a.imshow(drop(t, angle=270)); a.axis("off"); a.set_title(f"t={t}", fontsize=8)
a = fig.add_subplot(2, 1, 2)
a.plot(T, sweep, color=INK, marker="o", mfc=PAPER); a.axhline(487.5, color=INK2, ls="--", lw=1)
a.text(0, 470, "the best photo, +487.5", color=INK2, fontsize=9, va="top")
a.set_xlabel("sharpness t"); a.set_ylabel("4e:55 max"); a.set_ylim(-20, 520)
plt.tight_layout(); plt.show()

Sharper drives it harder, up to a point, so the drop does carry the signal. But the best drawing reaches less than a
quarter of what a real Pomeranian does. A point is part of what 4e:55 wants, not all of it: the lecture's whiskers and
pale fur fanning to a point (Plate X) are the rest.

### Your turn 2 · Turn the drop

The lecture also turned the sharpest drop (t = 1) through eight directions. Which way must the tip point? Score
`drop(1.0, angle=a)` for `a` in 0, 45, …, 315 and find the best angle.

*Hint: `score([drop(1.0, angle=a) for a in range(0, 360, 45)])`.*

In [ ]:
# TODO: score the drop at eight angles and print the best one

## Part B · The car that wasn't hiding

## Plate XII · Where the car goes

In layer 4c there is a car neuron, **4c:447**. The essay follows its wires into layer 4d and finds four neurons
(4d:417, 167, 295 and 241), two of which seem to fire on dogs as well as cars. Is the car hiding inside dog
neurons?

First the wires. Between 4c and 4d each neuron gets its input through a **branch**: either a single 1 × 1 kernel, or a
1 × 1 "bottleneck" kernel, a ReLU, then a 3 × 3 or 5 × 5 kernel. The **expanded weight** from one 4c neuron to one 4d
neuron multiplies the two kernels together and ignores the ReLU in between (as the *Zoom In* papers do). Summing it
over the kernel gives one number: how strongly, overall, the car pushes that 4d neuron.

The lecture: the four are the car's readers ranked **1, 4, 5 and 7** of 528, with sums **+0.767, +0.415, +0.411,
+0.398**.

In [ ]:
def branch_of(layer, ch):
    """Which branch a channel belongs to, and its index inside the branch."""
    s = 0
    for b in BRANCHES:
        n = getattr(net, f"{layer}_{b}_pre_relu_conv").out_channels
        if ch < s + n:
            return b, ch - s
        s += n


def expanded(layer, ch):
    """The kernel from every input channel straight to one channel: (inputs, k, k)."""
    b, j = branch_of(layer, ch)
    K = getattr(net, f"{layer}_{b}_pre_relu_conv").weight[j].cpu()                # (inputs or bottleneck, k, k)
    if b in ("3x3", "5x5"):
        B = getattr(net, f"{layer}_{b}_bottleneck_pre_relu_conv").weight[:, :, 0, 0].cpu()   # (bottleneck, inputs)
        K = torch.einsum("rkl,ri->ikl", K, B)                                  # multiply the two kernels
    return K.numpy()


n4d = sum(getattr(net, f"mixed4d_{b}_pre_relu_conv").out_channels for b in BRANCHES)
car = np.array([expanded("mixed4d", c)[447].sum() for c in range(n4d)])
rank = lambda c: int((car > car[c]).sum()) + 1
for c, s_, r_ in [(417, 0.7671, 1), (295, 0.4152, 4), (167, 0.4111, 5), (241, 0.3985, 7)]:
    print(f"4d:{c}  {branch_of('mixed4d', c)[0]} branch")
    check(f"  car -> 4d:{c}, summed expanded weight", car[c], s_, tol=1e-3)
    check(f"  its rank among the car's {n4d} readers", rank(c), r_)

In [ ]:
o = np.argsort(-car)[:20]
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(range(20), car[o], color=[INK if c in (417, 167, 295, 241) else PAPER for c in o], edgecolor=INK)
ax.set_xticks(range(20), [str(c) for c in o], rotation=90, fontsize=8)
ax.set_ylabel("car 4c:447 → 4d, summed"); ax.set_title("The car's 20 strongest wires into 4d (the four in black)", fontsize=10)
plt.tight_layout(); plt.show()

### Your turn 3 · Who reads 4d:241?

Do the same one layer further on: which neuron in 4e does 4d:241 push hardest? Use `expanded("mixed4e", e)[241].sum()`
for every 4e neuron `e` (there are 832).

*Hint: the lecture's answer is a car specialist, 71% cars among its top photos.*

In [ ]:
# TODO: find 4d:241's strongest reader in 4e

### Their photos

The lecture's Plate XII, step 4: what the four fire on. The lecture ranked all 1.28 million training photos; here we take
a slice of 3,000 validation photos (photos 0–2,999, a random mix of all 1,000 classes) and look at each neuron's 30
strongest. The checks compare with the lecture's own scan of these same 3,000 photos.

In [ ]:
fetch_range(0, 3000)
S = list(range(3000))
FOUR = [("mixed4d", c) for c in (417, 167, 295, 241)]
sl = measure(S, FOUR + [("mixed4c", 447)])
labels = np.array([label(i) for i in S])
LECTURE_SCAN = {417: [19, 8], 167: [18, 2], 295: [12, 17], 241: [12, 18]}     # (vehicles, dogs) among each one's top 30 in the lecture's scan of these photos
for w in FOUR:
    top = np.argsort(-sl[w]["max"])[:30]
    v = int(np.isin(labels[top], VEHICLES).sum()); d = int(np.isin(labels[top], list(DOGS)).sum())
    print(f"4d:{w[1]}: {v} vehicles, {d} dogs among its top 30")
    check(f"  4d:{w[1]} vehicles in top 30", v, LECTURE_SCAN[w[1]][0], tol=1)
    check(f"  4d:{w[1]} dogs in top 30", d, LECTURE_SCAN[w[1]][1], tol=1)

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(12, 3))
for row, ch in zip(axes, (295, 241)):
    top = np.argsort(-sl[("mixed4d", ch)]["max"])[:10]
    for a, j in zip(row, top):
        a.imshow(photo(S[j])); a.set_xticks([]); a.set_yticks([])
        a.set_title(NAMES[labels[j]][:14], fontsize=7)
    row[0].set_ylabel(f"4d:{ch}", fontsize=10)
plt.suptitle("The two mixed neurons' strongest photos: cars and dogs", fontsize=10); plt.tight_layout(); plt.show()

Two of the four (417 and 167) are boxy-car neurons; the other two (295 and 241) take cars and dogs. In the lecture's
1.28 million training photos: 4d:295 is 70% cars and 24% dogs, 4d:241 is 62% and 33%.

### Your turn 4 · The car's other pole

The lecture's gate (Plate XIX): the car neuron 4c:447 is pushed *down* by something specific. Using `sl[("mixed4c",
447)]["min"]`, find the 10 photos of the slice that push it lowest. What are they?

*Hint: `np.argsort(sl[("mixed4c", 447)]["min"])[:10]`, then `NAMES[labels[j]]`.*

In [ ]:
# TODO: the 10 photos that push car neuron 4c:447 lowest

## Plate XIII · Drawing down the car's road

**Feature visualization** draws a picture that drives a neuron as high as it will go: start from noise and follow the
gradient. Drawn freely, 4d:241 gives a dog. The essay's fix is to *guide* the drawing along the route real photos take:
for the first 64 of 512 steps, also push the car neuron 4c:447. The lecture found a light push was not enough, and a push
four times stronger made the drawings cars.

Below is the lecture's recipe in full (an image built from random frequencies, decorrelated colour, and small random
jitters, scales and turns each step, so the drawing can't cheat with pixel noise). We draw all four both ways (eight
drawings at once) and ask InceptionV1 what each drawing is. About two minutes on a T4.

In [ ]:
COLOR = np.array([[0.26, 0.09, 0.02], [0.27, 0.00, -0.05], [0.27, -0.09, 0.03]], dtype=np.float32)
COLOR = COLOR / np.max(np.linalg.norm(COLOR, axis=0))


def fft_image(B, h=224, w=224, sd=0.01, seed=0):
    """An image made of random frequencies, low ones louder: the 2017 Lucid parameterization."""
    g = torch.Generator().manual_seed(seed)
    fy, fx = np.fft.fftfreq(h)[:, None], np.fft.fftfreq(w)[: w // 2 + 1]
    freqs = np.sqrt(fx * fx + fy * fy)
    spec = (torch.randn((B, 3) + freqs.shape + (2,), generator=g) * sd).to(DEV).requires_grad_(True)
    scale = torch.tensor(1.0 / np.maximum(freqs, 1.0 / max(w, h))).float()[None, None, ..., None].to(DEV)
    C = torch.tensor(COLOR, device=DEV)
    def image():
        x = torch.fft.irfftn(torch.view_as_complex(scale * spec), s=(h, w), norm="ortho") / 4.0
        return torch.sigmoid(torch.einsum("bchw,dc->bdhw", x, C))
    return spec, image


def jiggle(x):
    """Pad, jitter, rescale, rotate, jitter: the drawing must work from slightly different views."""
    import torch.nn.functional as F
    B = x.shape[0]
    x = F.pad(x, (12, 12, 12, 12), value=0.5)
    def jitter(x, d):
        dx, dy = np.random.randint(0, d + 1, 2)
        return x[:, :, dy:x.shape[2] - (d - dy), dx:x.shape[3] - (d - dx)]
    x = jitter(x, 8)
    x = F.interpolate(x, scale_factor=float(np.random.choice([1 + (i - 5) / 50.0 for i in range(11)])),
                      mode="bilinear", align_corners=False)
    a = math.radians(float(np.random.choice(list(range(-10, 11)) + 5 * [0])))
    th = torch.tensor([[math.cos(a), -math.sin(a), 0], [math.sin(a), math.cos(a), 0]], device=x.device).float()
    x = F.grid_sample(x, F.affine_grid(th[None].expand(B, 2, 3), list(x.shape), align_corners=False),
                      padding_mode="border", align_corners=False)
    x = jitter(x, 4)
    if min(x.shape[-2:]) < 224:
        x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
    return x


# A typical strong value for each neuron (the 99.9th percentile of its per-photo max in the lecture's scan), so the
# target and the guide count on the same scale.
SCALE = {("mixed4d", 241): 717.5025, ("mixed4c", 447): 2056.006}


SCALE.update({("mixed4d", 417): 924.0, ("mixed4d", 167): 566.001, ("mixed4d", 295): 539.5005})


def draw(jobs, steps=512, guide_steps=64, seed=0):
    """jobs: list of (4d channel, guide weight). All drawn together, as the lecture did, each with its own image."""
    np.random.seed(seed); torch.manual_seed(seed)
    spec, image = fft_image(len(jobs), seed=seed)
    opt = torch.optim.Adam([spec], lr=0.05)
    for step in range(steps):
        opt.zero_grad()
        net(to_input(jiggle(image())))
        loss = 0.0
        for b, (ch, gw) in enumerate(jobs):
            loss = loss - taps["mixed4d"][b, ch].mean() / SCALE[("mixed4d", ch)]
            if step < guide_steps and gw:
                loss = loss - gw * taps["mixed4c"][b, 447].mean() / SCALE[("mixed4c", 447)]
        loss.backward(); opt.step()
    with torch.no_grad():
        x = image(); logits = net(to_input(x))
    return (x.permute(0, 2, 3, 1).cpu().numpy() * 255).astype(np.uint8), logits


# InceptionV1's own 1,008 labels (the 2014 TensorFlow release's order)
LUCID = requests.get("https://storage.googleapis.com/modelzoo/labels/ImageNet_alternate.txt", timeout=60).text.splitlines()
FOUR_CH = [417, 167, 295, 241]
t0 = time.time()
imgs, logits = draw([(c, 0.0) for c in FOUR_CH] + [(c, 4.0) for c in FOUR_CH])
print(f"drawn in {time.time() - t0:.0f} s")
says = [LUCID[int(l.argmax())] for l in logits]
VEHICLE_WORDS = {NAMES[c] for c in VEHICLES} | {"grille"}
LECTURE_FREE, LECTURE_CAR = ["Egyptian cat", "armadillo", "Lakeland terrier", "Irish terrier"], ["tow truck", "recreational vehicle", "grille", "pickup"]
for k, c in enumerate(FOUR_CH):
    print(f"4d:{c}  free: {says[k]:22s} (lecture: {LECTURE_FREE[k]:16s})  through the car: {says[4 + k]:22s} (lecture: {LECTURE_CAR[k]})")
n_free = sum(s_ in VEHICLE_WORDS for s_ in says[:4]); n_car = sum(s_ in VEHICLE_WORDS for s_ in says[4:])
with torch.no_grad():                                   # the car neuron on each finished drawing
    net(to_input(torch.from_numpy(imgs).permute(0, 3, 1, 2).float().to(DEV) / 255))
    car_on = taps["mixed4c"][:, 447].mean((1, 2)).cpu().numpy()
print("car neuron 4c:447 on the drawings, free:", minus(car_on[:4].round(0)), " guided:", minus(car_on[4:].round(0)))
# The plate's "all four" was one seed (0). The lecture's rerun over seeds 1-3 (logs/e3_fv_robust.json) gives 2 of 4
# guided drawings a vehicle's label in every seed, and 0 of 4 free: that is the number to check.
check("free drawings InceptionV1 calls a vehicle (of 4)", n_free, 0, tol=1)
check("guided drawings InceptionV1 calls a vehicle (of 4)", n_car, 2, tol=1)
check("every guided drawing lights the car neuron more than its free twin", bool((car_on[4:] > car_on[:4]).all()), True)

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(10, 5.4))
for k, c in enumerate(FOUR_CH):
    for r, lab_ in enumerate(("free", "through the car")):
        a = ax[r, k]; a.imshow(imgs[r * 4 + k]); a.axis("off")
        a.set_title(f"4d:{c} {lab_}\n{says[r * 4 + k]}", fontsize=9)
plt.tight_layout(); plt.show()

Drawn freely, the four fall into animals (cats, an armadillo, terriers), never vehicles. Drawn down the car's road, every
one of them carries the car: the car neuron lights up in all four. But by InceptionV1's own label only about half become
vehicles. 4d:417 is a tow truck every time and 4d:241 usually a pickup, while 4d:167 always comes out a "military
uniform". Seed 0, the run the lecture's plate shows, gave four of four. Rerun with three more seeds, the lecture's own code
gave two of four each time, and the plate now shows both; two of four is the number checked here. A drawing is a chaotic optimization: change the seed or
the hardware and a single picture can land somewhere else, so count several.

Nothing is hiding: the four are wired from the car and read cars, and two of them also read dogs. The question the essay then asks is *why* cars and dogs
together. Its answer: they share a **pose**.

## Part C · Do cars and dogs share a pose?

## Plates XV–XVII · Seeing pose

The essay's answer to "why cars *and* dogs?" is **pose**: 4d:241 fires on a car and a dog turned the same way to the
camera, and the essay counts more than a hundred such *generalized pose neurons* in InceptionV1.

The lecture's first three tests couldn't settle it. InceptionV1 barely tells a photo from its mirror image, so a mirror
test sees nothing, and the two judges we asked (CLIP and a vision-language model) almost never say "three-quarter view".

So we used an instrument built for the job: **Orient Anything** (Wang et al., 2024), a DINOv2 vision transformer
trained on renders of 3D models to read an object's **azimuth**: the angle it is turned through, 0° facing the
camera, 90° side-on, 180° facing away. A COCO object detector first boxes the car or the dog, and the regressor reads
the box.

InceptionV1 can't tell left from right, so we **fold** the angle: a car turned 40° to the left and one turned 40° to the
right both count as 40°. Folded, 0° is head-on, 90° profile, 180° from behind, and 30°–60° is a three-quarter view.

In [ ]:
# The detector (torchvision's COCO Faster R-CNN, 167 MB) and Orient Anything (1.2 GB of weights).
import torchvision
from huggingface_hub import hf_hub_download
from transformers import AutoImageProcessor, Dinov2Config, Dinov2Model

import contextlib
with contextlib.redirect_stderr(io.StringIO()):          # quiet download
    detector = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights="DEFAULT", progress=False).eval().to(DEV)


class Head(torch.nn.Module):
    """Orient Anything's read-out: two layers from DINOv2's summary token to 722 numbers
    (360 azimuth bins, 180 polar, 180 rotation, 2 for "does it have a front?")."""
    def __init__(self, i=1024, o=722):
        super().__init__()
        self.net1 = torch.nn.Sequential(torch.nn.Linear(i, o), torch.nn.BatchNorm1d(o), torch.nn.ReLU(inplace=True))
        self.net2 = torch.nn.Sequential(torch.nn.Linear(o, o), torch.nn.BatchNorm1d(o))
    def forward(self, x):
        return self.net2(self.net1(x))


class OrientAnything(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.dinov2 = Dinov2Model(Dinov2Config.from_pretrained("facebook/dinov2-large"))
        self.down_sampler = Head()
    def forward(self, px):
        return self.down_sampler(self.dinov2(pixel_values=px).last_hidden_state[:, 0])


oa = OrientAnything()
ck = hf_hub_download("Viglong/Orient-Anything", "croplargeEX2/dino_weight.pt")
oa.load_state_dict(torch.load(ck, map_location="cpu", weights_only=True)); oa = oa.eval().to(DEV)
prep = AutoImageProcessor.from_pretrained("facebook/dinov2-large")
print("loaded the COCO detector (167 MB) and Orient Anything (1.2 GB)")

In [ ]:
import random
VEH_COCO, DOG_COCO = {3, 4, 6, 7, 8}, {18}          # COCO's car, motorcycle, bus, train, truck; and dog


def box(img, kind):
    """The largest detected box of the right kind (score >= 0.5), or None."""
    with torch.no_grad():
        o = detector([torch.from_numpy(np.array(img)).permute(2, 0, 1).float().div(255).to(DEV)])[0]
    ok = VEH_COCO if kind == "vehicle" else DOG_COCO
    bx = [b.tolist() for b, l, s in zip(o["boxes"], o["labels"], o["scores"]) if int(l) in ok and float(s) >= 0.5]
    return max(bx, key=lambda b: (b[2] - b[0]) * (b[3] - b[1])) if bx else None


def square(img, b, pad=1.2):
    cx, cy, s = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2, pad * max(b[2] - b[0], b[3] - b[1])
    return Image.fromarray(img).crop((round(cx - s / 2), round(cy - s / 2), round(cx + s / 2), round(cy + s / 2)))


def circular_mean(a):
    r = np.deg2rad(a); x, y = np.cos(r), np.sin(r)
    d = np.hypot(x - x.mean(), y - y.mean()); q1, q3 = np.quantile(d, [0.25, 0.75])
    k = (d >= q1 - 1.5 * (q3 - q1)) & (d <= q3 + 1.5 * (q3 - q1))
    return float(np.rad2deg(np.arctan2(y[k].mean(), x[k].mean())) % 360)


def azimuth(crop, seed):
    """Read the crop five times (itself and four random 80-95% sub-crops); circular mean, as the authors do."""
    rng = random.Random(seed); views = [crop]; W_, H_ = crop.size
    for _ in range(4):
        w, h = rng.randint(int(W_ * .8), int(W_ * .95)), rng.randint(int(H_ * .8), int(H_ * .95))
        l, t = rng.randint(0, W_ - w), rng.randint(0, H_ - h); views.append(crop.crop((l, t, l + w, t + h)))
    px = torch.from_numpy(np.array(prep(images=views)["pixel_values"])).to(DEV)
    with torch.no_grad():
        p = oa(px).float()
    conf = float(torch.softmax(p[:, -2:], 1)[:, 0].mean())
    return circular_mean(p[:, :360].argmax(1).cpu().numpy().astype(float)), conf


def folded(az):
    """0 = facing the camera, 90 = profile, 180 = facing away; left and right merged."""
    return abs((az + 180) % 360 - 180)


def read_pose(ids, kind):
    out = []
    for i in ids:
        img = photo(i); b = box(img, kind)
        if b is None:
            out.append((i, None, None)); continue
        az, conf = azimuth(square(img, b), i)
        out.append((i, folded(az), conf))
    return out

### 4d:241's strongest cars and dogs

The lecture's run read every validation photo of a vehicle (1,250) and of a dog (5,900). Here we read only 4d:241's 20
strongest vehicle photos and 20 strongest dog photos, and compare their median angle with the lecture's.

In [ ]:
TOP241 = {"vehicle": [41872, 7373, 7843, 25468, 5808, 41919, 3111, 42415, 35787, 32337, 18767, 31952, 20753, 23095, 12489, 6083, 18394, 48443, 49888, 2919], "dog": [33356, 23556, 35297, 48529, 14258, 25393, 6270, 26420, 15857, 5523, 14150, 46211, 4559, 424, 41143, 10583, 7970, 11151, 34180, 5138]}
fetch(TOP241["vehicle"] + TOP241["dog"])
pose = {k: read_pose(TOP241[k], k) for k in ("vehicle", "dog")}
med = {k: float(np.median([f for _, f, c in pose[k] if f is not None])) for k in pose}
print(f"median folded angle: vehicles {med['vehicle']:.0f}°, dogs {med['dog']:.0f}°")
check("4d:241's top vehicles, median folded angle", med["vehicle"], 47.1, tol=8)
check("4d:241's top dogs, median folded angle", med["dog"], 5.5, tol=8)

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(12, 3.2))
for row, k in zip(axes, ("vehicle", "dog")):
    for a, (i, f, c) in zip(row, pose[k][:10]):
        a.imshow(photo(i)); a.set_xticks([]); a.set_yticks([])
        a.set_title("no box" if f is None else f"{f:.0f}°", fontsize=9)
    row[0].set_ylabel(k + "s", fontsize=10)
plt.suptitle("4d:241's strongest photos, with the folded angle Orient Anything reads", fontsize=10)
plt.tight_layout(); plt.show()

### Tuning: does the neuron *prefer* an angle, and the same one for both?

A top-20 list mixes up two things: what the neuron likes, and how ImageNet happens to photograph cars and dogs. (Dogs
are mostly photographed head-on.) The lecture's main test asked a sharper question of all 7,150 photos.

1. Take the neuron's value on every car photo, and subtract the average of that photo's own class. Now a class that is
   always photographed from one side can't fake a preference.
2. Average what is left in six bins of folded angle, 0–30°, 30–60°, …, 150–180°. That is the neuron's **tuning curve**
   for cars. Do the same for dogs.
3. If the neuron detects one pose across both, the two curves rise and fall together. Their correlation is the
   **agreement** r.

Here are the lecture's curves for the essay's two mixed neurons.

In [ ]:
CURVES = {"4d:241": {"T_v": [-0.1792, 0.3808, -0.0086, -0.0574, -0.0721, -0.7477], "T_d": [0.0838, -0.0304, -0.2955, -0.3185, -0.3671, -0.2659], "se_v": [0.0527, 0.0453, 0.1002, 0.0947, 0.0933, 0.0812], "se_d": [0.0246, 0.0522, 0.0636, 0.0663, 0.141, 0.2002], "r_vd": 0.25180674268428815, "p": 0.4275724275724276}, "4d:295": {"T_v": [-0.2045, 0.3937, -0.0938, -0.1688, 0.0538, -0.6977], "T_d": [0.0556, -0.0263, -0.2114, -0.1764, -0.1093, -0.3663], "se_v": [0.0529, 0.0445, 0.0963, 0.0957, 0.0939, 0.0754], "se_d": [0.0225, 0.0497, 0.0656, 0.0804, 0.155, 0.2482], "r_vd": 0.6722176018843168, "p": 0.17382617382617382}}
bins = np.arange(15, 180, 30)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for a, (n, c) in zip(ax, CURVES.items()):
    for k, ls, mk in (("vehicle", "-", "s"), ("dog", "--", "o")):
        a.errorbar(bins, c["T_" + k[0]], yerr=c["se_" + k[0]], color=INK, ls=ls, marker=mk, mfc=PAPER, capsize=3, label=k + "s")
    a.axhline(0, color=INK2, lw=0.8)
    a.set_title(f"{n}: agreement r = {c['r_vd']:+.2f} (p = {c['p']:.3f})", fontsize=10)
    a.set_xticks([0, 30, 60, 90, 120, 150, 180]); a.set_xlabel("folded angle (0 = head-on, 90 = profile)")
ax[0].set_ylabel("value, minus its class's mean (sd units)"); ax[0].legend()
plt.tight_layout(); plt.show()

### The whole population

The lecture computed the agreement for the 54 neurons whose top photos mix vehicles and dogs (the essay's candidates), and
for 200 random neurons from the same layers as a control: if *every* neuron's curves agreed, agreement would say nothing
about pose neurons.

In [ ]:
R_CROSS = [-0.0443, -0.1276, -0.1231, 0.6743, 0.1048, -0.1015, 0.702, -0.2452, 0.5113, 0.6654, 0.2518, 0.7551, 0.6073, 0.6722, 0.7153, 0.3379, 0.7934, -0.1504, 0.8075, -0.3535, 0.0972, -0.0102, 0.569, 0.0168, 0.4874, 0.4845, 0.6002, 0.2835, 0.0854, 0.1527, 0.5616, 0.859, 0.2604, 0.8138, 0.1065, 0.6, 0.3503, -0.7172, 0.2679, -0.2925, 0.2015, -0.099, 0.6326, 0.6355, 0.8542, 0.1263, -0.141, 0.8336, -0.4659, 0.6544, -0.0891, -0.4827, -0.3616, 0.1509]
R_RANDOM = [0.1374, -0.0118, 0.1203, -0.3882, 0.7687, -0.0898, 0.1887, 0.2978, -0.4036, 0.1934, 0.4611, -0.361, 0.3046, -0.3842, 0.3555, -0.3318, 0.71, 0.1317, -0.2924, 0.1205, -0.8694, 0.0485, 0.4191, 0.3208, 0.1364, -0.6125, -0.1969, 0.1349, -0.0928, 0.197, 0.219, -0.3281, -0.3411, 0.6189, 0.0593, 0.2125, 0.7384, -0.8011, 0.2901, 0.6256, 0.4629, 0.3985, 0.6437, -0.5285, 0.6902, 0.522, -0.8605, 0.657, -0.2389, 0.8215, 0.3095, 0.0267, -0.1644, 0.8002, 0.4636, 0.3545, -0.5228, -0.1717, 0.5055, 0.4726, -0.7282, 0.6311, 0.656, 0.2307, 0.4084, 0.2876, 0.7204, 0.2921, -0.8877, 0.2969, -0.3197, 0.7217, 0.2399, -0.3091, -0.3074, 0.0034, -0.2821, 0.0642, -0.5173, -0.0057, -0.4441, -0.7772, -0.214, -0.1484, 0.5382, -0.2454, -0.0253, 0.6613, -0.2109, -0.6881, -0.1352, -0.0732, -0.6221, 0.0138, 0.7359, -0.2551, 0.581, -0.3296, 0.3297, 0.2276, -0.0772, 0.0074, 0.8727, 0.782, 0.1064, 0.7298, 0.2222, -0.0665, -0.1866, -0.3895, 0.8264, 0.23, -0.1973, -0.4001, 0.6152, -0.4139, -0.1937, -0.6759, -0.6823, 0.5182, 0.3796, -0.1511, -0.3118, 0.2661, -0.1263, -0.0076, 0.5242, 0.3276, 0.5389, -0.0745, -0.9116, -0.2625, 0.2282, -0.4378, -0.3423, 0.3134, -0.2983, 0.5225, -0.6591, 0.3275, 0.2962, -0.4366, -0.0777, 0.3966, -0.4013, -0.8573, 0.1404, 0.2735, 0.2501, 0.7253, -0.3119, -0.6153, -0.3692, -0.7641, -0.0037, 0.3958, 0.3041, -0.0498, -0.7661, 0.5941, -0.4883, 0.8567, 0.1952, 0.6711, -0.0513, -0.3527, -0.2902, 0.1578, -0.0883, -0.492, -0.6851, 0.6649, 0.4916, -0.9405, -0.2449, -0.2899, -0.606, -0.1736, -0.6188, 0.1483, 0.1931, 0.0343, -0.2695, -0.7612, 0.018, -0.0245, -0.8186, -0.5413, -0.5234, -0.8127, 0.4833, -0.7723, -0.4785, -0.9239, 0.6126, -0.7166, -0.2054, 0.7288, 0.4798, 0.1452]
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(R_RANDOM, bins=np.linspace(-1, 1, 21), color=PAPER, edgecolor=INK2, density=True, label="200 random neurons")
ax.hist(R_CROSS, bins=np.linspace(-1, 1, 21), histtype="step", color=INK, lw=2, density=True, label="54 car-and-dog neurons")
ax.set_xlabel("agreement r between the car and the dog tuning curves"); ax.legend(); plt.tight_layout(); plt.show()
print(f"median r: car-and-dog neurons {np.median(R_CROSS):+.2f}, random {np.median(R_RANDOM):+.2f}")
check("median agreement, car-and-dog neurons", np.median(R_CROSS), 0.2642, tol=1e-3)
check("median agreement, random neurons", np.median(R_RANDOM), 0.0054, tol=1e-3)

### Your turn 5 · Is the difference real?

Are the car-and-dog neurons' agreements higher than the random neurons', or could the gap be luck? Write a **rank test**
by hand (Mann–Whitney): pool all 254 values, rank them, and add up the ranks of the 54. Then shuffle which values are
called "car-and-dog" 10,000 times and see how often a shuffle beats the real sum. That share is the p-value.

*Hint: `scipy.stats.rankdata`, then `np.random.permutation`.*

In [ ]:
# TODO: a permutation rank test of R_CROSS against R_RANDOM

### What the lecture concluded

The test was written down before it ran, with a rule for calling the result (Plate XVII):

1. the car-and-dog neurons must agree more than random neurons: **yes** (median r +0.26 against
   +0.01);
2. at least half of them must agree on their own: **no** (11 of 54);
3. the essay's two examples must each agree: **no** (4d:241 p = 0.43, 4d:295 p =
   0.17).

So the verdict is **partly**. And one more control changes the reading: neurons that read only cars agree just as much
(median +0.29). Cars and dogs do share a rough pose in InceptionV1, front towards the
camera, but it belongs to every neuron that reads cars, not to a special set of pose detectors. The top photos you
measured above show it too: 4d:241's strongest cars are three-quarter views, its strongest dogs head-on.

## Plates XVIII–XXIV · What else the lecture tested

These need bigger runs than a notebook should make, so here are the results with where they come from:

- **XVIII · Other networks.** Looking for the same car-and-dog neuron: the public ConvNeXt-T has 4 such units of 23,040;
  DINOv2 ViT-B has 27 of 24,576, the clearest pairing head-on car grilles with head-on dog faces.
- **XIX · A neuron is a gate.** Car neuron 4c:447's negative pole is fluffy dogs (Your turn 4): it keeps dogs out of
  the car path.
- **XX · Sibling poles.** A neuron's two poles are close relatives only slightly more often than chance.
- **XXI · The dog basin.** Free drawings of random deep neurons fall into dogs rarely (2 of 54); most are unnameable.
- **XXII · Three microscopes.** The same neuron drawn three ways gives three different pictures.
- **XXIII · Umbrella → bee.** Retraining InceptionV1 to call umbrellas bees reroutes them through bee neurons; at gentle
  learning rates the change is spread through the network.
- **XXIV · Weak activations.** Zeroing the weakest values of one layer costs little accuracy (under a point).

## Solutions

**Your turn 1 · The dog test.**

In [ ]:
snout = measure(POINTED_SNOUT_IDS, [N55])[N55]["max"]
flat = measure(FLAT_FACED_IDS, [N55])[N55]["max"]
print(f"{len(snout)} pointed-snout photos, {len(flat)} flat-faced")
check("AUC, pointed snouts over flat faces (10 per breed)", auc(snout, flat), 0.691, tol=0.08)

With 10 photos per breed the AUC lands near the lecture's 0.691 (all 1,400 photos): the dog test passes, unlike the cat test.

**Your turn 2 · Turn the drop.**

In [ ]:
angles = list(range(0, 360, 45))
turned = score([drop(1.0, angle=a) for a in angles])
for a, v in zip(angles, turned):
    print(f"angle {a:3d}: {v:6.1f}")
check("best direction for the tip", angles[int(np.argmax(turned))], 270)
check("its value", turned.max(), 108.39, tol=1.5)

The tip pointing right (270) wins, and pointing left (90) is close behind; pointing up (180) is worst. 4e:55 cares which way a point faces in the picture.

**Your turn 3 · Who reads 4d:241?**

In [ ]:
n4e = sum(getattr(net, f"mixed4e_{b}_pre_relu_conv").out_channels for b in BRANCHES)
readers = np.array([expanded("mixed4e", e)[241].sum() for e in range(n4e)])
o = np.argsort(-readers)[:3]
print("4d:241's strongest readers in 4e:", [(int(e), round(float(readers[e]), 3)) for e in o])
check("its strongest reader", int(o[0]), 591)
check("that wire's summed weight", readers[o[0]], 0.3064, tol=1e-3)

4e:591, a car specialist (71% cars among its top 1,000 training photos): the car signal flows on.

**Your turn 4 · The car's other pole.**

In [ ]:
low = np.argsort(sl[("mixed4c", 447)]["min"])[:10]
print([NAMES[labels[j]] for j in low])
check("dogs among the 10 photos that push the car lowest", int(np.isin(labels[low], list(DOGS)).sum()), 5, tol=1)

Furry dogs and dog-like animals (wolves, dingoes): the car neuron is pushed *down* by them, a gate that keeps dogs out of the car path.

**Your turn 5 · Is the difference real?**

In [ ]:
from scipy.stats import rankdata
vals = np.array(R_CROSS + R_RANDOM); n = len(R_CROSS)
ranks = rankdata(vals)
real = ranks[:n].sum()
rng = np.random.default_rng(0)
beats = sum(ranks[rng.permutation(len(vals))[:n]].sum() >= real for _ in range(10000))
p = (beats + 1) / 10001
print(f"rank sum {real:.0f}; shuffles that beat it: {beats} of 10,000; p = {p:.4f}")
from scipy.stats import mannwhitneyu
check("p, by shuffling, vs the lecture's Mann-Whitney p", p, 0.0002, tol=max(0.01, 0.0002 * 0.5))